# load_from_ddb

DynamoDB에서 conf/data를 내려받아 로컬 `conf/`, `data/` 디렉토리를 생성합니다.
`modeling.ipynb` 실행 전 준비 단계.

## 1. 환경 설정

In [1]:
import sys
import yaml
from pathlib import Path

BASE_DIR = Path.cwd()
sys.path.insert(0, str(BASE_DIR))
from ddb_store import DDBStore
print(f'Base: {BASE_DIR}')

Base: /home/ec2-user/SageMaker/gs-ds-env/samples/hjsong/modeling


## 2. 실험 식별 정보

In [2]:
REGION     = 'us-east-1'
USER_ID    = 'hjsong'
PROJECT    = 'titanic-survival-prediction'
EXPERIMENT = 'tuned-hjsong-v1'

store  = DDBStore(region=REGION)
EXP_PK = DDBStore.make_exp_pk(USER_ID, PROJECT, EXPERIMENT)
print(f'EXP_PK: {EXP_PK}')

EXP_PK: EXP#hjsong#titanic-survival-prediction#tuned-hjsong-v1


## 3. conf/ 생성

In [3]:
CONF_DIR = BASE_DIR / 'conf'
CONF_DIR.mkdir(parents=True, exist_ok=True)

conf = store.get_experiment_conf(EXP_PK)

for name, data in [('env',conf['env_yml']),('meta',conf['meta_yml']),('model',conf['model_yml'])]:
    path = CONF_DIR / f'{name}.yml'
    with open(path,'w',encoding='utf-8') as f:
        yaml.dump(data, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
    print(f'  [OK] {path.name}: {path.stat().st_size}bytes')

print('conf/ 생성 완료')

  [OK] env.yml: 749bytes
  [OK] meta.yml: 894bytes
  [OK] model.yml: 1413bytes
conf/ 생성 완료


## 4. data/ 생성

In [4]:
DATA_DIR = BASE_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

for split in ['train', 'validation', 'test']:
    df   = store.get_dataset_split(EXP_PK, split)
    path = DATA_DIR / f'{split}.csv'
    df.to_csv(path, index=False)
    print(f'  [OK] {path.name}: {len(df)}rows, {path.stat().st_size/1024:.1f}KB')

print('data/ 생성 완료')

  [OK] train.csv: 891rows, 60.5KB
  [OK] validation.csv: 418rows, 28.3KB
  [OK] test.csv: 418rows, 28.3KB
data/ 생성 완료


## 5. 검증

In [5]:
print('=' * 60)
for p in [CONF_DIR/'env.yml', CONF_DIR/'meta.yml', CONF_DIR/'model.yml',
          DATA_DIR/'train.csv', DATA_DIR/'validation.csv', DATA_DIR/'test.csv']:
    ok = p.exists()
    print(f'  {"OK" if ok else "MISSING"}: {p.name} ({p.stat().st_size/1024:.1f}KB)' if ok else f'  MISSING: {p.name}')
print('=' * 60)
print('다음 단계: modeling.ipynb 또는 modeling_ddb.ipynb')

  OK: env.yml (0.7KB)
  OK: meta.yml (0.9KB)
  OK: model.yml (1.4KB)
  OK: train.csv (60.5KB)
  OK: validation.csv (28.3KB)
  OK: test.csv (28.3KB)
다음 단계: modeling.ipynb 또는 modeling_ddb.ipynb
